In [0]:
# ============================================================
# Bronze — Source 10: Partner S3 Drop
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=10_partner_s3/
# Target: bronze.partner catalog in Unity Catalog
#
# Tables:
#   - bronze.partner.sales
#
# Raw format: Parquet / Avro (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "10_partner_s3"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/")

In [0]:
# Cell 2 — Read Parquet and Avro files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path_parquet = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.parquet"
path_avro = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.avro"

df_parquet = spark.read.format("parquet") \
    .load(path_parquet) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

df_avro = spark.read.format("avro") \
    .load(path_avro) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Parquet rows: {df_parquet.count()}")
print(f"Avro rows: {df_avro.count()}")
df_parquet.printSchema()

In [0]:
# Cell 3 — Union Parquet + Avro and write to Bronze
df_combined = df_parquet.union(df_avro)
print(f"Combined rows: {df_combined.count()}")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.partner")

df_combined.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.partner.sales")

latest_ts = df_parquet.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, df_combined.count())
print(f"✅ bronze.partner.sales: {df_combined.count()} rows (Parquet + Avro)")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.partner.sales").collect()[0]['cnt']
print(f"bronze.partner.sales: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '10_partner_s3'").show()